In [0]:
# ============================================================
# PROYECTO FINAL ELECTROCASA
# 15. GOLD - MODELO ANALITICO
# 01. VALIDACION DE FUENTES SILVER
# ============================================================

CATALOGO = "proyecto_final"

fuentes_silver = [
    "sucursales",
    "catalogo",
    "empleados",
    "resenas",
    "ventas",
    "devoluciones",
    "tracking_envios"
]

resumen = []

for fuente in fuentes_silver:
    tabla = f"{CATALOGO}.silver.{fuente}"
    df = spark.table(tabla)

    resumen.append((
        fuente,
        tabla,
        df.count(),
        len(df.columns)
    ))

    print(f"\n{'=' * 65}")
    print(f"FUENTE: {fuente.upper()}")
    print(f"TABLA: {tabla}")
    print("ESQUEMA:")
    df.printSchema()

df_resumen = spark.createDataFrame(
    resumen,
    ["fuente", "tabla_silver", "total_registros", "total_columnas"]
)

print("\n===== RESUMEN DE FUENTES SILVER =====")

display(df_resumen.orderBy("fuente"))

In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 02. CREACION DE FACT_VENTAS
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.fact_ventas
USING DELTA
AS
SELECT
    venta_id,
    sucursal_id,
    producto_id,
    vendedor_id,
    cantidad,
    monto_total,
    metodo_pago,
    canal,
    fecha_venta,

    CAST(fecha_venta AS DATE) AS fecha,

    DATE_TRUNC('MONTH', fecha_venta) AS mes_venta,

    YEAR(fecha_venta) AS anio,

    MONTH(fecha_venta) AS mes,

    CASE
        WHEN canal = 'tienda_fisica' THEN 1
        ELSE 0
    END AS es_tienda_fisica,

    CASE
        WHEN canal = 'ecommerce' THEN 1
        ELSE 0
    END AS es_ecommerce

FROM proyecto_final.silver.ventas

WHERE fecha_venta >= TIMESTAMP '2026-03-01 00:00:00'
  AND fecha_venta < TIMESTAMP '2026-11-01 00:00:00';


In [0]:

%sql
SELECT
    COUNT(*) AS total_ventas,
    COUNT(DISTINCT venta_id) AS ventas_distintas,
    COUNT(DISTINCT sucursal_id) AS sucursales,
    COUNT(DISTINCT producto_id) AS productos,
    SUM(cantidad) AS unidades_vendidas,
    ROUND(SUM(monto_total), 2) AS ventas_brutas,
    MIN(fecha_venta) AS primera_venta,
    MAX(fecha_venta) AS ultima_venta
FROM proyecto_final.gold.fact_ventas;


In [0]:
%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 04. CREACION DE FACT_DEVOLUCIONES
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.fact_devoluciones
USING DELTA
AS
SELECT
    d.devolucion_id,
    d.venta_id,
    d.sucursal_id,
    d.producto_id,
    d.motivo,
    d.monto_reembolso,
    d.fecha_devolucion,

    DATE_TRUNC(
        'MONTH',
        CAST(d.fecha_devolucion AS TIMESTAMP)
    ) AS mes_devolucion,

    YEAR(d.fecha_devolucion) AS anio,
    MONTH(d.fecha_devolucion) AS mes,

    -- Información de la venta original
    v.fecha_venta,
    v.monto_total AS monto_venta_original,
    v.canal AS canal_venta

FROM proyecto_final.silver.devoluciones d

INNER JOIN proyecto_final.gold.fact_ventas v
    ON d.venta_id = v.venta_id

WHERE d.fecha_devolucion >= DATE '2026-03-01'
  AND d.fecha_devolucion < DATE '2026-11-01';

In [0]:
%sql
-- ============================================================
-- 05. VALIDACION DE FACT_DEVOLUCIONES
-- ============================================================

SELECT
    COUNT(*) AS total_devoluciones,
    COUNT(DISTINCT devolucion_id) AS devoluciones_distintas,
    COUNT(DISTINCT venta_id) AS ventas_con_devolucion,
    COUNT(DISTINCT sucursal_id) AS sucursales,
    ROUND(SUM(monto_reembolso), 2) AS total_reembolsado,
    MIN(fecha_devolucion) AS primera_devolucion,
    MAX(fecha_devolucion) AS ultima_devolucion,

    SUM(
        CASE
            WHEN monto_reembolso > monto_venta_original
            THEN 1 ELSE 0
        END
    ) AS reembolsos_excesivos

FROM proyecto_final.gold.fact_devoluciones;

In [0]:
%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 06. VENTAS NETAS POR SUCURSAL Y MES
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.ventas_netas_mensuales
USING DELTA
AS

WITH ventas_mensuales AS (
    SELECT
        sucursal_id,
        CAST(mes_venta AS DATE) AS periodo,
        COUNT(DISTINCT venta_id) AS cantidad_ventas,
        SUM(monto_total) AS ventas_brutas
    FROM proyecto_final.gold.fact_ventas
    GROUP BY
        sucursal_id,
        CAST(mes_venta AS DATE)
),

devoluciones_mensuales AS (
    SELECT
        sucursal_id,
        CAST(mes_devolucion AS DATE) AS periodo,
        COUNT(DISTINCT devolucion_id) AS cantidad_devoluciones,
        SUM(monto_reembolso) AS total_devoluciones
    FROM proyecto_final.gold.fact_devoluciones
    GROUP BY
        sucursal_id,
        CAST(mes_devolucion AS DATE)
)

SELECT
    COALESCE(v.sucursal_id, d.sucursal_id) AS sucursal_id,
    COALESCE(v.periodo, d.periodo) AS periodo,

    COALESCE(v.cantidad_ventas, 0) AS cantidad_ventas,
    COALESCE(d.cantidad_devoluciones, 0) AS cantidad_devoluciones,

    CAST(COALESCE(v.ventas_brutas, 0) AS DECIMAL(18,2))
        AS ventas_brutas,

    CAST(COALESCE(d.total_devoluciones, 0) AS DECIMAL(18,2))
        AS total_devoluciones,

    CAST(
        COALESCE(v.ventas_brutas, 0)
        - COALESCE(d.total_devoluciones, 0)
        AS DECIMAL(18,2)
    ) AS ventas_netas,

    CAST(
        CASE
            WHEN COALESCE(v.cantidad_ventas, 0) > 0
            THEN v.ventas_brutas / v.cantidad_ventas
            ELSE NULL
        END
        AS DECIMAL(18,2)
    ) AS ticket_promedio

FROM ventas_mensuales v

FULL OUTER JOIN devoluciones_mensuales d
    ON v.sucursal_id = d.sucursal_id
   AND v.periodo = d.periodo;

In [0]:
%sql
-- ============================================================
-- 07. VALIDACION VENTAS NETAS MENSUALES
-- ============================================================

SELECT
    COUNT(*) AS total_sucursal_mes,
    COUNT(DISTINCT sucursal_id) AS sucursales,
    COUNT(DISTINCT periodo) AS meses,

    SUM(cantidad_ventas) AS total_ventas,
    SUM(cantidad_devoluciones) AS total_devoluciones,

    ROUND(SUM(ventas_brutas), 2) AS ventas_brutas,
    ROUND(SUM(total_devoluciones), 2) AS monto_devoluciones,
    ROUND(SUM(ventas_netas), 2) AS ventas_netas,

    SUM(
        CASE
            WHEN ventas_netas <> ventas_brutas - total_devoluciones
            THEN 1 ELSE 0
        END
    ) AS registros_descuadrados

FROM proyecto_final.gold.ventas_netas_mensuales;

In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- GOLD - PREGUNTA 1
-- VENTAS NETAS POR SUCURSAL Y MES
-- ============================================================

SELECT
    g.periodo,
    g.sucursal_id,
    s.nombre_sucursal,
    s.ciudad,
    g.cantidad_ventas,
    g.cantidad_devoluciones,
    g.ventas_brutas,
    g.total_devoluciones,
    g.ventas_netas,
    g.ticket_promedio

FROM proyecto_final.gold.ventas_netas_mensuales g

LEFT JOIN proyecto_final.silver.sucursales s
    ON g.sucursal_id = s.sucursal_id

ORDER BY
    g.periodo,
    g.ventas_netas DESC,
    g.sucursal_id

LIMIT 50;


In [0]:
%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 09. METRICAS DE PRODUCTOS
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.productos_metricas
USING DELTA
AS

WITH ventas_producto AS (
    SELECT
        producto_id,
        COUNT(DISTINCT venta_id) AS cantidad_ventas,
        SUM(cantidad) AS unidades_vendidas,
        SUM(monto_total) AS monto_vendido
    FROM proyecto_final.gold.fact_ventas
    GROUP BY producto_id
),

devoluciones_producto AS (
    SELECT
        producto_id,
        COUNT(DISTINCT devolucion_id) AS cantidad_devoluciones,
        SUM(monto_reembolso) AS monto_reembolsado
    FROM proyecto_final.gold.fact_devoluciones
    GROUP BY producto_id
)

SELECT
    v.producto_id,
    v.cantidad_ventas,
    v.unidades_vendidas,
    CAST(v.monto_vendido AS DECIMAL(18,2)) AS monto_vendido,

    COALESCE(d.cantidad_devoluciones, 0)
        AS cantidad_devoluciones,

    CAST(
        COALESCE(d.monto_reembolsado, 0)
        AS DECIMAL(18,2)
    ) AS monto_reembolsado,

    CAST(
        COALESCE(d.cantidad_devoluciones, 0)
        * 100.0 / v.cantidad_ventas
        AS DECIMAL(10,2)
    ) AS tasa_devolucion_pct

FROM ventas_producto v

LEFT JOIN devoluciones_producto d
    ON v.producto_id = d.producto_id;

In [0]:
%sql
-- ============================================================
-- 10. VALIDACION DE METRICAS DE PRODUCTOS
-- ============================================================

SELECT
    COUNT(*) AS productos_analizados,
    SUM(cantidad_ventas) AS total_ventas,
    SUM(unidades_vendidas) AS total_unidades,
    SUM(cantidad_devoluciones) AS total_devoluciones,

    ROUND(SUM(monto_vendido), 2) AS monto_vendido,
    ROUND(SUM(monto_reembolsado), 2) AS monto_reembolsado,

    SUM(
        CASE
            WHEN tasa_devolucion_pct < 0
              OR tasa_devolucion_pct > 100
            THEN 1 ELSE 0
        END
    ) AS tasas_fuera_rango

FROM proyecto_final.gold.productos_metricas;

In [0]:
%sql
-- ============================================================
-- GOLD - PREGUNTA 2A
-- TOP 10 PRODUCTOS POR UNIDADES VENDIDAS
-- ============================================================

SELECT
    producto_id,
    unidades_vendidas,
    cantidad_ventas,
    monto_vendido,
    cantidad_devoluciones,
    tasa_devolucion_pct

FROM proyecto_final.gold.productos_metricas

ORDER BY
    unidades_vendidas DESC,
    cantidad_ventas DESC,
    producto_id

LIMIT 10;

In [0]:
%sql
-- ============================================================
-- GOLD - PREGUNTA 2B
-- TOP 10 PRODUCTOS CON MAYOR TASA DE DEVOLUCION
-- ============================================================

SELECT
    producto_id,
    cantidad_ventas,
    cantidad_devoluciones,
    unidades_vendidas,
    tasa_devolucion_pct,
    monto_reembolsado

FROM proyecto_final.gold.productos_metricas

ORDER BY
    tasa_devolucion_pct DESC,
    cantidad_devoluciones DESC,
    cantidad_ventas DESC,
    producto_id

LIMIT 10;

In [0]:
%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 13. EMPLEADOS ACTIVOS AL CIERRE DE CADA MES - SCD2
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.fact_empleados_mes
USING DELTA
AS

WITH meses AS (
    SELECT
        EXPLODE(
            SEQUENCE(
                DATE '2026-03-01',
                DATE '2026-10-01',
                INTERVAL 1 MONTH
            )
        ) AS periodo
),

cierres AS (
    SELECT
        periodo,
        LAST_DAY(periodo) AS fecha_cierre
    FROM meses
)

SELECT
    c.periodo,
    c.fecha_cierre,

    e.id_empleado,
    e.sucursal_id,
    e.cargo,
    e.version,
    e.vigente_desde,
    e.vigente_hasta

FROM cierres c

INNER JOIN proyecto_final.silver.empleados e

    ON e.vigente_desde <= c.fecha_cierre

   AND (
        e.vigente_hasta IS NULL
        OR e.vigente_hasta > c.fecha_cierre
   )

   AND e.activo = TRUE;

In [0]:
%sql
-- ============================================================
-- 14. VALIDACION DE EMPLEADOS POR MES
-- ============================================================

SELECT
    periodo,
    fecha_cierre,
    COUNT(*) AS total_empleados_activos,
    COUNT(DISTINCT id_empleado) AS empleados_distintos,
    COUNT(DISTINCT sucursal_id) AS sucursales,

    COUNT(*) - COUNT(DISTINCT id_empleado)
        AS empleados_duplicados

FROM proyecto_final.gold.fact_empleados_mes

GROUP BY periodo, fecha_cierre
ORDER BY periodo;

In [0]:
%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 15. DOTACION MENSUAL POR SUCURSAL
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.dotacion_mensual
USING DELTA
AS

WITH meses AS (
    SELECT EXPLODE(
        SEQUENCE(
            DATE '2026-03-01',
            DATE '2026-10-01',
            INTERVAL 1 MONTH
        )
    ) AS periodo
),

dotacion AS (
    SELECT
        periodo,
        sucursal_id,
        COUNT(DISTINCT id_empleado) AS empleados_activos
    FROM proyecto_final.gold.fact_empleados_mes
    GROUP BY periodo, sucursal_id
)

SELECT
    m.periodo,
    LAST_DAY(m.periodo) AS fecha_cierre,
    s.sucursal_id,
    s.nombre_sucursal,
    s.ciudad,
    s.region,
    COALESCE(d.empleados_activos, 0) AS empleados_activos

FROM meses m
CROSS JOIN proyecto_final.silver.sucursales s

LEFT JOIN dotacion d
    ON m.periodo = d.periodo
   AND s.sucursal_id = d.sucursal_id;

In [0]:
%sql
-- ============================================================
-- GOLD - PREGUNTA 3
-- EMPLEADOS ACTIVOS AL CIERRE DE CADA MES
-- ============================================================

SELECT
    periodo,
    fecha_cierre,
    sucursal_id,
    nombre_sucursal,
    ciudad,
    empleados_activos

FROM proyecto_final.gold.dotacion_mensual

ORDER BY
    periodo,
    empleados_activos DESC,
    sucursal_id

LIMIT 50;

In [0]:
%sql
-- ============================================================
-- 17. CONCILIACION DOTACION GOLD VS HISTORIAL SCD2
-- ============================================================

WITH historico AS (
    SELECT
        periodo,
        COUNT(DISTINCT id_empleado) AS total_scd2
    FROM proyecto_final.gold.fact_empleados_mes
    GROUP BY periodo
),

dotacion AS (
    SELECT
        periodo,
        COUNT(*) AS sucursales,
        SUM(empleados_activos) AS total_gold
    FROM proyecto_final.gold.dotacion_mensual
    GROUP BY periodo
)

SELECT
    d.periodo,
    d.sucursales,
    h.total_scd2,
    d.total_gold,
    d.total_gold - h.total_scd2 AS diferencia,

    CASE
        WHEN d.sucursales = 40
         AND d.total_gold = h.total_scd2
        THEN 'OK'
        ELSE 'ERROR'
    END AS estado_validacion

FROM dotacion d
INNER JOIN historico h
    ON d.periodo = h.periodo

ORDER BY d.periodo;

In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- GOLD - 18. VENTAS CON VENDEDOR Y SUCURSAL HISTORICA
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.fact_ventas_vendedor
USING DELTA
AS

WITH ventas_fisicas AS (
    SELECT *
    FROM proyecto_final.gold.fact_ventas
    WHERE canal = 'tienda_fisica'
),

ventas_historicas AS (
    SELECT
        v.venta_id,
        v.vendedor_id,
        v.sucursal_id AS sucursal_venta,
        v.producto_id,
        v.cantidad,
        v.monto_total,
        v.fecha_venta,
        CAST(v.mes_venta AS DATE) AS periodo,

        e.sucursal_id AS sucursal_historica,
        e.vigente_desde,
        e.vigente_hasta,

        ROW_NUMBER() OVER (
            PARTITION BY v.venta_id
            ORDER BY
                e.vigente_desde DESC NULLS LAST,
                e.version DESC NULLS LAST
        ) AS numero_fila

    FROM ventas_fisicas v

    LEFT JOIN proyecto_final.silver.empleados e
        ON v.vendedor_id = e.id_empleado
       AND CAST(v.fecha_venta AS DATE) >= e.vigente_desde
       AND (
           e.vigente_hasta IS NULL
           OR CAST(v.fecha_venta AS DATE) < e.vigente_hasta
       )
       AND e.activo = TRUE
)

SELECT
    venta_id,
    vendedor_id,
    sucursal_venta,

    COALESCE(
        sucursal_historica,
        'SIN_ASIGNAR'
    ) AS sucursal_historica,

    producto_id,
    cantidad,
    monto_total,
    fecha_venta,
    periodo,

    CASE
        WHEN sucursal_historica IS NULL THEN 0
        ELSE 1
    END AS tiene_historial_valido

FROM ventas_historicas
WHERE numero_fila = 1;


In [0]:

%sql
-- ============================================================
-- GOLD - 19. VALIDACION VENTAS POR VENDEDOR HISTORICO
-- ============================================================

SELECT
    COUNT(*) AS total_ventas_fisicas,
    COUNT(DISTINCT venta_id) AS ventas_distintas,
    COUNT(DISTINCT vendedor_id) AS vendedores_distintos,

    ROUND(SUM(monto_total), 2) AS monto_ventas_fisicas,

    SUM(
        CASE WHEN sucursal_historica = 'SIN_ASIGNAR'
        THEN 1 ELSE 0 END
    ) AS ventas_sin_asignacion,

    SUM(
        CASE WHEN sucursal_historica <> 'SIN_ASIGNAR'
        THEN 1 ELSE 0 END
    ) AS ventas_con_asignacion,

    SUM(
        CASE
            WHEN sucursal_historica <> 'SIN_ASIGNAR'
             AND sucursal_historica <> sucursal_venta
            THEN 1 ELSE 0
        END
    ) AS sucursales_diferentes_origen

FROM proyecto_final.gold.fact_ventas_vendedor;


In [0]:

%sql
-- ============================================================
-- GOLD - PREGUNTA 4
-- VENTAS FISICAS POR VENDEDOR, SUCURSAL HISTORICA Y MES
-- ============================================================

SELECT
    v.periodo,
    v.vendedor_id,
    v.sucursal_historica AS sucursal_id,

    COALESCE(
        s.nombre_sucursal,
        'SIN_ASIGNAR'
    ) AS nombre_sucursal,

    COUNT(DISTINCT v.venta_id) AS cantidad_ventas,
    ROUND(SUM(v.monto_total), 2) AS monto_vendido

FROM proyecto_final.gold.fact_ventas_vendedor v

LEFT JOIN proyecto_final.silver.sucursales s
    ON v.sucursal_historica = s.sucursal_id

GROUP BY
    v.periodo,
    v.vendedor_id,
    v.sucursal_historica,
    s.nombre_sucursal

ORDER BY
    v.periodo,
    monto_vendido DESC,
    v.vendedor_id

LIMIT 50;


In [0]:

# ============================================================
# PROYECTO FINAL ELECTROCASA
# GOLD - 21. ESTRUCTURAS PARA ANALISIS DE RESENAS
# ============================================================

tablas = [
    "proyecto_final.silver.resenas",
    "proyecto_final.silver.catalogo",
    "proyecto_final.bronze.catalogo"
]

for tabla in tablas:
    df = spark.table(tabla)

    print("\n" + "=" * 65)
    print("TABLA:", tabla)
    print("COLUMNAS:", df.columns)
    print("TOTAL REGISTROS:", df.count())
    df.printSchema()

    if tabla == "proyecto_final.bronze.catalogo":
        display(df.limit(5))


In [0]:

%sql
-- ============================================================
-- GOLD - 22. IDENTIFICACION DE SNAPSHOTS DEL CATALOGO
-- ============================================================

SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT producto_id) AS productos_distintos,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.catalogo
GROUP BY archivo_origen
ORDER BY archivo_origen;


In [0]:

%sql
-- ============================================================
-- GOLD - 23. ESTRUCTURA DE RESENAS SILVER
-- ============================================================

DESCRIBE TABLE proyecto_final.silver.resenas;


In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- GOLD - 24. DIMENSION DE PRODUCTOS POR ULTIMO SNAPSHOT
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.dim_productos
USING DELTA
AS

WITH catalogo_historico AS (
    SELECT
        TRIM(producto_id) AS producto_id,
        TRIM(nombre_producto) AS nombre_producto,
        TRIM(categoria) AS categoria_original,
        TRIM(marca) AS marca,

        TO_DATE(
            REGEXP_EXTRACT(
                archivo_origen,
                '(20[0-9]{2}-[0-9]{2}-[0-9]{2})',
                1
            ),
            'yyyy-MM-dd'
        ) AS fecha_snapshot,

        fecha_ingesta,
        archivo_origen

    FROM proyecto_final.bronze.catalogo

    WHERE producto_id IS NOT NULL
      AND TRIM(producto_id) <> ''
),

catalogo_normalizado AS (
    SELECT
        *,

        CASE
            WHEN LOWER(categoria_original) IN (
                'electrónica', 'electronica'
            ) THEN 'Electrónica'

            WHEN LOWER(categoria_original) IN (
                'línea blanca', 'linea blanca'
            ) THEN 'Línea blanca'

            WHEN LOWER(categoria_original) = 'cocina'
                THEN 'Cocina'

            WHEN LOWER(categoria_original) = 'computación'
                OR LOWER(categoria_original) = 'computacion'
                THEN 'Computación'

            ELSE INITCAP(categoria_original)
        END AS categoria

    FROM catalogo_historico
),

versiones AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY producto_id
            ORDER BY
                fecha_snapshot DESC NULLS LAST,
                fecha_ingesta DESC NULLS LAST,
                archivo_origen DESC,
                nombre_producto ASC NULLS LAST,
                categoria ASC NULLS LAST
        ) AS numero_fila

    FROM catalogo_normalizado
)

SELECT
    producto_id,
    nombre_producto,
    categoria,
    marca,
    fecha_snapshot

FROM versiones
WHERE numero_fila = 1;


In [0]:

%sql
-- ============================================================
-- GOLD - 25. VALIDACION DIMENSION DE PRODUCTOS
-- ============================================================

SELECT
    (SELECT COUNT(*)
     FROM proyecto_final.gold.dim_productos)
        AS total_productos,

    (SELECT COUNT(DISTINCT producto_id)
     FROM proyecto_final.gold.dim_productos)
        AS productos_distintos,

    (SELECT COUNT(*)
     FROM proyecto_final.gold.dim_productos
     WHERE fecha_snapshot = DATE '2026-08-01')
        AS productos_snapshot_agosto,

    (SELECT COUNT(*)
     FROM proyecto_final.gold.dim_productos
     WHERE fecha_snapshot = DATE '2026-03-01')
        AS productos_solo_marzo,

    (SELECT COUNT(*)
     FROM proyecto_final.silver.resenas r
     LEFT JOIN proyecto_final.gold.dim_productos p
       ON r.producto_id = p.producto_id
     WHERE p.producto_id IS NULL)
        AS resenas_sin_categoria;


In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 26. RESENAS NEGATIVAS POR CATEGORIA Y MES
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.resenas_negativas_mensuales
USING DELTA
AS

WITH resenas_clasificadas AS (
    SELECT
        r.resena_id,
        r.producto_id,
        r.calificacion,
        r.fecha_resena,

        CAST(
            DATE_TRUNC('MONTH', r.fecha_resena)
            AS DATE
        ) AS periodo,

        COALESCE(
            p.categoria,
            'SIN_CATEGORIA'
        ) AS categoria,

        CASE
            WHEN r.calificacion IN (1, 2)
            THEN 1
            ELSE 0
        END AS es_negativa

    FROM proyecto_final.silver.resenas r

    LEFT JOIN proyecto_final.gold.dim_productos p
        ON r.producto_id = p.producto_id

    WHERE r.fecha_resena >= DATE '2026-03-01'
      AND r.fecha_resena < DATE '2026-11-01'
)

SELECT
    periodo,
    categoria,

    COUNT(*) AS total_resenas,

    SUM(es_negativa) AS resenas_negativas,

    COUNT(*) - SUM(es_negativa)
        AS resenas_no_negativas,

    CAST(
        SUM(es_negativa) * 100.0 / COUNT(*)
        AS DECIMAL(10,2)
    ) AS porcentaje_negativas

FROM resenas_clasificadas

GROUP BY periodo, categoria;


In [0]:

%sql
-- ============================================================
-- GOLD - 27. VALIDACION RESENAS NEGATIVAS
-- ============================================================

SELECT
    COUNT(*) AS combinaciones_categoria_mes,
    COUNT(DISTINCT periodo) AS meses,
    COUNT(DISTINCT categoria) AS categorias,

    SUM(total_resenas) AS total_resenas,
    SUM(resenas_negativas) AS total_negativas,
    SUM(resenas_no_negativas) AS total_no_negativas,

    CAST(
        SUM(resenas_negativas) * 100.0
        / NULLIF(SUM(total_resenas), 0)
        AS DECIMAL(10,2)
    ) AS porcentaje_negativas_global,

    SUM(
        CASE
            WHEN porcentaje_negativas < 0
              OR porcentaje_negativas > 100
            THEN 1 ELSE 0
        END
    ) AS porcentajes_invalidos,

    SUM(
        CASE
            WHEN total_resenas <>
                 resenas_negativas + resenas_no_negativas
            THEN 1 ELSE 0
        END
    ) AS registros_descuadrados

FROM proyecto_final.gold.resenas_negativas_mensuales;


In [0]:

%sql
-- ============================================================
-- GOLD - PREGUNTA 5
-- PORCENTAJE DE RESENAS NEGATIVAS POR CATEGORIA Y MES
-- ============================================================

SELECT
    periodo,
    categoria,
    total_resenas,
    resenas_negativas,
    resenas_no_negativas,
    porcentaje_negativas

FROM proyecto_final.gold.resenas_negativas_mensuales

ORDER BY
    periodo,
    porcentaje_negativas DESC,
    categoria;


In [0]:

%sql
-- ============================================================
-- GOLD - 29. CORRECCION DE CATEGORIAS
-- ============================================================

UPDATE proyecto_final.gold.dim_productos
SET categoria =
    CASE
        WHEN LOWER(REPLACE(categoria, '_', ' '))
             IN ('linea blanca', 'línea blanca')
            THEN 'Línea blanca'

        WHEN LOWER(REPLACE(categoria, '_', ' '))
             IN ('climatizacion', 'climatización')
            THEN 'Climatización'

        ELSE categoria
    END;


In [0]:

%sql
-- ============================================================
-- GOLD - 30. RECONSTRUCCION RESENAS POR CATEGORIA
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.resenas_negativas_mensuales
USING DELTA
AS

SELECT
    CAST(DATE_TRUNC('MONTH', r.fecha_resena) AS DATE)
        AS periodo,

    COALESCE(p.categoria, 'SIN_CATEGORIA')
        AS categoria,

    COUNT(*) AS total_resenas,

    SUM(
        CASE WHEN r.calificacion IN (1, 2)
        THEN 1 ELSE 0 END
    ) AS resenas_negativas,

    SUM(
        CASE WHEN r.calificacion NOT IN (1, 2)
        THEN 1 ELSE 0 END
    ) AS resenas_no_negativas,

    CAST(
        100.0 * SUM(
            CASE WHEN r.calificacion IN (1, 2)
            THEN 1 ELSE 0 END
        ) / COUNT(*)
        AS DECIMAL(10,2)
    ) AS porcentaje_negativas

FROM proyecto_final.silver.resenas r

LEFT JOIN proyecto_final.gold.dim_productos p
    ON r.producto_id = p.producto_id

WHERE r.fecha_resena >= DATE '2026-03-01'
  AND r.fecha_resena < DATE '2026-11-01'

GROUP BY
    CAST(DATE_TRUNC('MONTH', r.fecha_resena) AS DATE),
    COALESCE(p.categoria, 'SIN_CATEGORIA');


In [0]:

%sql
-- ============================================================
-- GOLD - 31. VALIDACION FINAL RESENAS
-- ============================================================

SELECT
    COUNT(*) AS combinaciones_categoria_mes,
    COUNT(DISTINCT periodo) AS meses,
    COUNT(DISTINCT categoria) AS categorias,
    SUM(total_resenas) AS total_resenas,
    SUM(resenas_negativas) AS negativas,
    SUM(resenas_no_negativas) AS no_negativas,
    ROUND(
        SUM(resenas_negativas) * 100.0 /
        SUM(total_resenas), 2
    ) AS porcentaje_negativas_global

FROM proyecto_final.gold.resenas_negativas_mensuales;


In [0]:

%sql
-- ============================================================
-- GOLD - 32. PREGUNTA 5 FINAL
-- ============================================================

SELECT
    periodo,
    categoria,
    total_resenas,
    resenas_negativas,
    resenas_no_negativas,
    porcentaje_negativas

FROM proyecto_final.gold.resenas_negativas_mensuales

ORDER BY
    periodo,
    porcentaje_negativas DESC,
    categoria;


In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 33. FACT ENVIOS CON HISTORIAL DE ESTADOS
-- ============================================================

CREATE OR REPLACE TABLE proyecto_final.gold.fact_envios
USING DELTA
AS

WITH eventos AS (
    SELECT
        evento_id,
        tracking_id,
        venta_id,
        courier,
        estado_entrega,
        sucursal_origen,
        fecha_modificacion
    FROM proyecto_final.silver.tracking_envios
),

primeros_eventos AS (
    SELECT
        tracking_id,

        MIN(
            CASE WHEN estado_entrega = 'pendiente'
            THEN fecha_modificacion END
        ) AS primer_pendiente,

        MIN(
            CASE WHEN estado_entrega = 'entregado'
            THEN fecha_modificacion END
        ) AS primer_entregado,

        MIN(
            CASE WHEN estado_entrega = 'devuelto'
            THEN fecha_modificacion END
        ) AS primer_devuelto,

        COUNT(*) AS total_eventos

    FROM eventos
    GROUP BY tracking_id
),

ultimo_estado AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY tracking_id
            ORDER BY fecha_modificacion DESC, evento_id DESC
        ) AS numero_fila
    FROM eventos
)

SELECT
    u.tracking_id,
    u.venta_id,
    u.courier,
    u.sucursal_origen,

    u.estado_entrega AS estado_final,
    u.fecha_modificacion AS fecha_ultimo_estado,

    p.primer_pendiente,
    p.primer_entregado,
    p.primer_devuelto,
    p.total_eventos,

    CASE
        WHEN u.estado_entrega = 'entregado'
         AND p.primer_pendiente IS NOT NULL
         AND p.primer_entregado >= p.primer_pendiente
        THEN
            CAST(
                (
                    UNIX_TIMESTAMP(p.primer_entregado)
                    - UNIX_TIMESTAMP(p.primer_pendiente)
                ) / 3600.0
                AS DECIMAL(12,2)
            )
        ELSE NULL
    END AS horas_entrega,

    CASE
        WHEN u.estado_entrega IN ('entregado', 'devuelto')
        THEN 1 ELSE 0
    END AS tiene_estado_final,

    CASE
        WHEN u.estado_entrega = 'devuelto'
        THEN 1 ELSE 0
    END AS es_devuelto

FROM ultimo_estado u

INNER JOIN primeros_eventos p
    ON u.tracking_id = p.tracking_id

WHERE u.numero_fila = 1;


In [0]:

%sql
-- ============================================================
-- GOLD - 34. VALIDACION DE FACT ENVIOS
-- ============================================================

SELECT
    COUNT(*) AS total_envios,
    COUNT(DISTINCT tracking_id) AS envios_distintos,

    SUM(total_eventos) AS eventos_procesados,

    SUM(CASE
        WHEN estado_final = 'entregado'
        THEN 1 ELSE 0 END
    ) AS envios_entregados,

    SUM(CASE
        WHEN estado_final = 'devuelto'
        THEN 1 ELSE 0 END
    ) AS envios_devueltos,

    SUM(CASE
        WHEN estado_final = 'pendiente'
        THEN 1 ELSE 0 END
    ) AS envios_pendientes,

    SUM(CASE
        WHEN estado_final = 'en_transito'
        THEN 1 ELSE 0 END
    ) AS envios_en_transito,

    SUM(CASE
        WHEN horas_entrega IS NOT NULL
        THEN 1 ELSE 0 END
    ) AS entregas_tiempo_calculable,

    SUM(CASE
        WHEN horas_entrega < 0
        THEN 1 ELSE 0 END
    ) AS tiempos_negativos,

    ROUND(AVG(horas_entrega), 2)
        AS horas_promedio_entrega

FROM proyecto_final.gold.fact_envios;


In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- GOLD - PREGUNTA 6
-- 35. TIEMPO DE ENTREGA Y PORCENTAJE DE DEVOLUCION
-- ============================================================

SELECT
    COUNT(*) AS total_envios,

    SUM(
        CASE WHEN estado_final = 'entregado'
        THEN 1 ELSE 0 END
    ) AS envios_entregados,

    SUM(
        CASE WHEN estado_final = 'devuelto'
        THEN 1 ELSE 0 END
    ) AS envios_devueltos,

    SUM(tiene_estado_final) AS envios_con_estado_final,

    COUNT(horas_entrega) AS entregas_tiempo_calculable,

    ROUND(AVG(horas_entrega), 2)
        AS promedio_horas_entrega,

    ROUND(MIN(horas_entrega), 2)
        AS minimo_horas_entrega,

    ROUND(MAX(horas_entrega), 2)
        AS maximo_horas_entrega,

    CAST(
        100.0 * SUM(es_devuelto)
        / NULLIF(SUM(tiene_estado_final), 0)
        AS DECIMAL(10,2)
    ) AS porcentaje_envios_devueltos,

    SUM(
        CASE WHEN horas_entrega < 0
        THEN 1 ELSE 0 END
    ) AS tiempos_negativos

FROM proyecto_final.gold.fact_envios;


In [0]:

%sql
-- ============================================================
-- GOLD - 36. DETALLE DE TIEMPOS DE ENTREGA
-- ============================================================

SELECT
    tracking_id,
    venta_id,
    courier,
    sucursal_origen,
    estado_final,
    primer_pendiente,
    primer_entregado,
    fecha_ultimo_estado,
    horas_entrega,
    total_eventos

FROM proyecto_final.gold.fact_envios

WHERE estado_final = 'entregado'
  AND horas_entrega IS NOT NULL

ORDER BY horas_entrega DESC, tracking_id

LIMIT 30;


In [0]:

%sql
-- ============================================================
-- PROYECTO FINAL ELECTROCASA
-- 15. GOLD - MODELO ANALITICO
-- 37. INVENTARIO FINAL DE TABLAS GOLD
-- ============================================================

SELECT '01_fact_ventas' AS tabla,
       COUNT(*) AS total_registros
FROM proyecto_final.gold.fact_ventas

UNION ALL

SELECT '02_fact_devoluciones',
       COUNT(*)
FROM proyecto_final.gold.fact_devoluciones

UNION ALL

SELECT '03_ventas_netas_mensuales',
       COUNT(*)
FROM proyecto_final.gold.ventas_netas_mensuales

UNION ALL

SELECT '04_productos_metricas',
       COUNT(*)
FROM proyecto_final.gold.productos_metricas

UNION ALL

SELECT '05_fact_empleados_mes',
       COUNT(*)
FROM proyecto_final.gold.fact_empleados_mes

UNION ALL

SELECT '06_dotacion_mensual',
       COUNT(*)
FROM proyecto_final.gold.dotacion_mensual

UNION ALL

SELECT '07_fact_ventas_vendedor',
       COUNT(*)
FROM proyecto_final.gold.fact_ventas_vendedor

UNION ALL

SELECT '08_dim_productos',
       COUNT(*)
FROM proyecto_final.gold.dim_productos

UNION ALL

SELECT '09_resenas_negativas_mensuales',
       COUNT(*)
FROM proyecto_final.gold.resenas_negativas_mensuales

UNION ALL

SELECT '10_fact_envios',
       COUNT(*)
FROM proyecto_final.gold.fact_envios

ORDER BY tabla;
